In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [21]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [22]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Pascal Siakam,Over,25.5,-137,2025-11-20,2025-11-19T22:33:48Z
1,PrizePicks,player_points,Pascal Siakam,Under,25.5,-137,2025-11-20,2025-11-19T22:33:48Z
2,PrizePicks,player_points,LaMelo Ball,Over,22.5,-137,2025-11-20,2025-11-19T22:33:48Z
3,PrizePicks,player_points,LaMelo Ball,Under,22.5,-137,2025-11-20,2025-11-19T22:33:48Z
4,PrizePicks,player_points,Miles Bridges,Over,22.5,-137,2025-11-20,2025-11-19T22:33:48Z


### Update projected starting lineups

In [23]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Top EVs for single bets

In [24]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 109 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
0,Josh Giddey,BetRivers,21.5,25.27,Over,120,0,5.06,0.421,High
1,Pelle Larsson,BetRivers,10.5,13.81,Over,112,0,4.92,0.439,High
2,Aaron Gordon,BetRivers,19.5,22.79,Over,120,0,4.69,0.391,High
3,Josh Giddey,BetRivers,20.5,25.27,Over,102,1,4.51,0.442,High
4,Davion Mitchell,BetRivers,10.5,13.07,Over,115,0,4.44,0.386,High


## Top EVs for 2 leg bets

### Underdog picks

In [25]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 87 players...
Error getting prediction for Coby White: float division by zero
Processing 78 players with valid predictions...
Generated 2843 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 116 combinations from 2843 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Dereck Lively II,Isaac Okoro,4.5,5.5,7.09,9.04,over,over,0,5.94,0.297,Low,High
1,Aaron Gordon,Isaac Okoro,17.5,5.5,22.79,9.04,over,over,0,5.90,0.295,High,High
2,Landry Shamet,Isaac Okoro,9.5,5.5,13.84,9.04,over,over,0,5.88,0.294,High,High
3,Dereck Lively II,Josh Giddey,4.5,19.5,7.09,25.27,over,over,0,5.64,0.282,Low,High
4,Landry Shamet,Josh Giddey,9.5,19.5,13.84,25.27,over,over,0,5.57,0.279,High,High
5,Aaron Gordon,Dereck Lively II,17.5,4.5,22.79,7.09,over,over,0,5.56,0.278,High,Low
6,Aaron Gordon,Josh Giddey,17.5,19.5,22.79,25.27,over,over,1,5.55,0.277,High,High
7,Landry Shamet,Jerami Grant,9.5,23.5,13.84,18.55,over,under,0,4.99,0.250,High,High
8,Jerami Grant,Kevin Huerter,23.5,10.5,18.55,14.35,under,over,0,4.52,0.226,High,High
9,Kevin Huerter,Caleb Love,10.5,13.5,14.35,9.63,over,under,0,4.10,0.205,High,High


### Prizepicks picks

In [26]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 123 players...
Error getting prediction for Coby White: float division by zero
Processing 115 players with valid predictions...
Generated 6231 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 171 combinations from 6231 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Josh Giddey,Kris Murray,19.5,9.5,25.27,5.49,over,under,0,6.12,0.306,High,Low
1,Kris Murray,Isaac Okoro,9.5,5.5,5.49,9.04,under,over,0,5.94,0.297,Low,High
2,Dereck Lively II,Josh Giddey,4.5,19.5,7.09,25.27,over,over,0,5.87,0.294,Low,High
3,Dereck Lively II,Isaac Okoro,4.5,5.5,7.09,9.04,over,over,0,5.86,0.293,Low,High
4,Brandin Podziemski,Kris Murray,16.5,9.5,11.51,5.49,under,under,0,5.82,0.291,High,Low


## 3 leg parlay

### Underdog picks

In [27]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 87 players...
Error getting prediction for Coby White: float division by zero
Processing 78 players with valid predictions...
Generated 75094 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 51 combinations from 75094 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Dereck Lively II,Josh Giddey,Isaac Okoro,4.5,19.5,5.5,7.09,25.27,9.04,over,over,over,0,11.89,0.238,Low,High,High
1,Aaron Gordon,Josh Giddey,Isaac Okoro,17.5,19.5,5.5,22.79,25.27,9.04,over,over,over,0,11.81,0.236,High,High,High
2,Aaron Gordon,Dereck Lively II,Landry Shamet,17.5,4.5,9.5,22.79,7.09,13.84,over,over,over,0,11.21,0.224,High,Low,High
3,Landry Shamet,Jerami Grant,Caleb Love,9.5,23.5,13.5,13.84,18.55,9.63,over,under,under,0,9.55,0.191,High,High,High
4,Jeremiah Fears,Jerami Grant,Caleb Love,14.5,23.5,13.5,18.00,18.55,9.63,over,under,under,0,7.79,0.156,High,High,High


### Prizepicks picks

In [28]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 123 players...
Error getting prediction for Coby White: float division by zero
Processing 115 players with valid predictions...
Generated 244242 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 76 combinations from 244242 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Dereck Lively II,Kris Murray,Isaac Okoro,4.5,9.5,5.5,7.09,5.49,9.04,over,under,over,0,11.93,0.239,Low,Low,High
1,Dereck Lively II,Josh Giddey,Isaac Okoro,4.5,19.5,5.5,7.09,25.27,9.04,over,over,over,0,11.89,0.238,Low,High,High
2,Aaron Gordon,Josh Giddey,Kris Murray,17.5,19.5,9.5,22.79,25.27,5.49,over,over,under,0,11.61,0.232,High,High,Low
3,Brandin Podziemski,Aaron Gordon,Landry Shamet,16.5,17.5,9.5,11.51,22.79,13.84,under,over,over,0,11.12,0.222,High,High,High
4,Brandin Podziemski,Pelle Larsson,Landry Shamet,16.5,9.5,9.5,11.51,13.81,13.84,under,over,over,0,10.91,0.218,High,High,High
